# NSE 5× Turnaround Breakout — No Hard Stop + Portfolio Backtest

Full Colab notebook for the latest 5× turnaround research.

Changes:
- Removes the fixed -30% initial stop completely.
- Uses a configurable trailing stop, failure exit, and time exit.
- Adds a portfolio-level backtest with capital, cash, max positions, sizing, slippage and costs.
- Reports portfolio CAGR, drawdown, Sharpe, Sortino, Calmar, yearly returns and exposure.
- Includes trade-level diagnostics, trailing-stop sensitivity, Monte Carlo and walk-forward periods.
- Default data path: `/content/drive/MyDrive/quant/data/parquet`.

In [ ]:
# ============================================================
# 1. CONFIGURATION
# ============================================================
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/quant/data/parquet")
RESULTS_DIR = DATA_DIR.parent / "results" / "5x_turnaround"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

INITIAL_CAPITAL = 1_000_000.0
MAX_POSITIONS = 20
ENTRY_SCORE_MIN = 8
SIGNAL_COOLDOWN_DAYS = 57

# IMPORTANT: there is NO hard initial stop.
TRAILING_STOP_PCT = 0.30
FAILURE_EXIT_DAYS = 20
TIME_EXIT_DAYS = 252

# Approximate round-trip trading costs.
SLIPPAGE_BPS = 10
BROKERAGE_BPS = 0
STT_BPS_SELL = 10
OTHER_COST_BPS = 5

USE_REGIME_FILTER = False

print("DATA_DIR:", DATA_DIR)
print("RESULTS_DIR:", RESULTS_DIR)
print("Hard initial stop: DISABLED")

In [ ]:
# ============================================================
# 2. INSTALL / IMPORT
# ============================================================
import sys, subprocess, warnings, math, os, json
warnings.filterwarnings("ignore")

for package in ["duckdb", "pyarrow", "pandas", "numpy", "matplotlib", "scipy"]:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

print("DuckDB:", duckdb.__version__)

In [ ]:
# ============================================================
# 3. GOOGLE DRIVE
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception as e:
    print("Drive mount skipped:", e)

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory does not exist: {DATA_DIR}")

print("Data directory exists:", DATA_DIR)

In [ ]:
# ============================================================
# 4. DISCOVER PARQUET FILES
# ============================================================
files = sorted(DATA_DIR.rglob("*.parquet"))
print("Parquet files:", len(files))

if not files:
    raise FileNotFoundError(f"No parquet files found under {DATA_DIR}")

for f in files[:15]:
    print(f)

In [ ]:
# ============================================================
# 5. DUCKDB + SCHEMA
# ============================================================
con = duckdb.connect()

sample_file = str(files[0])
schema_df = con.execute(
    "DESCRIBE SELECT * FROM read_parquet(?)",
    [sample_file]
).df()

display(schema_df)

In [ ]:
# ============================================================
# 6. NORMALIZE DAILY PRICE VIEW
# ============================================================
def pick_col(columns, candidates):
    lower_map = {str(c).lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

orig_cols = schema_df["column_name"].tolist()

DATE_COL = pick_col(orig_cols, ["date", "trade_date", "datetime"])
SYMBOL_COL = pick_col(orig_cols, ["symbol", "ticker", "security", "scrip"])
OPEN_COL = pick_col(orig_cols, ["open", "open_price"])
HIGH_COL = pick_col(orig_cols, ["high", "high_price"])
LOW_COL = pick_col(orig_cols, ["low", "low_price"])
CLOSE_COL = pick_col(orig_cols, ["close", "close_price", "last", "ltp"])
VOL_COL = pick_col(orig_cols, ["volume", "qty", "quantity", "shares_traded", "total_traded_quantity"])

mapping = {
    "date": DATE_COL, "symbol": SYMBOL_COL, "open": OPEN_COL,
    "high": HIGH_COL, "low": LOW_COL, "close": CLOSE_COL, "volume": VOL_COL
}
print(mapping)

missing = [k for k, v in mapping.items() if v is None]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

root = str(DATA_DIR).replace("'", "''")

con.execute(f"""
CREATE OR REPLACE VIEW daily AS
SELECT
    CAST("{DATE_COL}" AS DATE) AS date,
    CAST("{SYMBOL_COL}" AS VARCHAR) AS symbol,
    TRY_CAST("{OPEN_COL}" AS DOUBLE) AS open,
    TRY_CAST("{HIGH_COL}" AS DOUBLE) AS high,
    TRY_CAST("{LOW_COL}" AS DOUBLE) AS low,
    TRY_CAST("{CLOSE_COL}" AS DOUBLE) AS close,
    TRY_CAST("{VOL_COL}" AS DOUBLE) AS volume
FROM read_parquet('{root}/**/*.parquet', union_by_name=true)
WHERE "{DATE_COL}" IS NOT NULL
""")

display(con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT symbol) AS total_symbols,
    MIN(date) AS first_date,
    MAX(date) AS last_date
FROM daily
""").df())

In [ ]:
# ============================================================
# 7. DATA QUALITY
# ============================================================
display(con.execute("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE close IS NULL OR close <= 0) AS bad_close,
    COUNT(*) FILTER (WHERE open IS NULL OR high IS NULL OR low IS NULL) AS bad_ohlc,
    COUNT(*) FILTER (WHERE volume IS NULL OR volume < 0) AS bad_volume,
    COUNT(DISTINCT symbol) AS symbols,
    MIN(date) AS first_date,
    MAX(date) AS last_date
FROM daily
""").df())

## 8. Feature engine

The strategy looks for depressed stocks that begin stabilizing and break above their prior 60-day high.

The current-day breakout is compared with the **prior** 60-day high, and the trade enters at the next trading day's open.

In [ ]:
# ============================================================
# 8. FEATURE TABLE
# ============================================================
con.execute("DROP TABLE IF EXISTS features")

con.execute("""
CREATE TABLE features AS
WITH base AS (
    SELECT
        date, symbol, open, high, low, close, volume,

        LAG(close, 20) OVER w AS close_20,
        LAG(close, 60) OVER w AS close_60,
        LAG(close, 120) OVER w AS close_120,
        LAG(close, 252) OVER w AS close_252,

        AVG(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 19 PRECEDING AND CURRENT ROW
        ) AS sma20,

        AVG(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 49 PRECEDING AND CURRENT ROW
        ) AS sma50,

        AVG(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 99 PRECEDING AND CURRENT ROW
        ) AS sma100,

        AVG(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 199 PRECEDING AND CURRENT ROW
        ) AS sma200,

        MAX(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
        ) AS high252,

        MIN(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 251 PRECEDING AND CURRENT ROW
        ) AS low252,

        MAX(close) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 60 PRECEDING AND 1 PRECEDING
        ) AS prior_high60,

        AVG(volume) OVER (
            PARTITION BY symbol ORDER BY date
            ROWS BETWEEN 59 PRECEDING AND CURRENT ROW
        ) AS avg_volume60
    FROM daily
    WINDOW w AS (PARTITION BY symbol ORDER BY date)
),
x AS (
    SELECT *,
        close / NULLIF(close_20, 0) - 1 AS ret20,
        close / NULLIF(close_60, 0) - 1 AS ret60,
        close / NULLIF(close_120, 0) - 1 AS ret120,
        close / NULLIF(close_252, 0) - 1 AS ret252,
        close / NULLIF(high252, 0) - 1 AS dist252high,
        (close - low252) / NULLIF(high252 - low252, 0) AS range_position,
        volume / NULLIF(avg_volume60, 0) AS volume_ratio60
    FROM base
),
z AS (
    SELECT *,
        CASE WHEN close > LAG(close, 5) OVER (
            PARTITION BY symbol ORDER BY date
        ) THEN 1 ELSE 0 END AS higher_than_5ago,

        CASE WHEN close > prior_high60 THEN 1 ELSE 0 END AS breakout60,

        CASE WHEN sma20 > LAG(sma20) OVER (
            PARTITION BY symbol ORDER BY date
        ) THEN 1 ELSE 0 END AS sma20_rising,

        CASE WHEN close > sma50 THEN 1 ELSE 0 END AS above_sma50,

        CASE WHEN sma50 > sma100 THEN 1 ELSE 0 END AS sma50_above100,

        CASE WHEN volume_ratio60 >= 1.5 THEN 1 ELSE 0 END AS volume_expansion
    FROM x
)
SELECT *,
    (
        CASE WHEN ret60 < -0.10 THEN 1 ELSE 0 END +
        CASE WHEN ret120 < -0.15 THEN 1 ELSE 0 END +
        CASE WHEN ret252 < -0.25 THEN 1 ELSE 0 END +
        CASE WHEN dist252high < -0.30 THEN 1 ELSE 0 END +
        CASE WHEN range_position < 0.50 THEN 1 ELSE 0 END +
        CASE WHEN higher_than_5ago = 1 THEN 1 ELSE 0 END +
        CASE WHEN sma20_rising = 1 THEN 1 ELSE 0 END +
        CASE WHEN breakout60 = 1 THEN 1 ELSE 0 END +
        CASE WHEN above_sma50 = 1 THEN 1 ELSE 0 END +
        CASE WHEN volume_expansion = 1 THEN 1 ELSE 0 END
    ) AS score
FROM z
WHERE close > 0
""")

display(con.execute("""
SELECT
    COUNT(*) AS feature_rows,
    COUNT(DISTINCT symbol) AS symbols,
    MIN(date) AS first_date,
    MAX(date) AS last_date
FROM features
""").df())

In [ ]:
# ============================================================
# 9. SIGNALS
# ============================================================
signals = con.execute(f"""
WITH s AS (
    SELECT
        date AS signal_date,
        symbol,
        open AS signal_open,
        close AS signal_close,
        high AS signal_high,
        low AS signal_low,
        volume AS signal_volume,
        score,
        ret20, ret60, ret120, ret252,
        dist252high,
        range_position,
        volume_ratio60,
        sma20, sma50, sma100, sma200,
        breakout60,
        sma20_rising,
        above_sma50,
        sma50_above100,
        volume_expansion,
        LEAD(date) OVER (
            PARTITION BY symbol ORDER BY date
        ) AS entry_date,
        LEAD(open) OVER (
            PARTITION BY symbol ORDER BY date
        ) AS entry_open
    FROM features
    WHERE score >= {int(ENTRY_SCORE_MIN)}
      AND breakout60 = 1
      AND sma20_rising = 1
      AND above_sma50 = 1
)
SELECT *
FROM s
WHERE entry_open IS NOT NULL
""").df()

signals["signal_date"] = pd.to_datetime(signals["signal_date"])
signals["entry_date"] = pd.to_datetime(signals["entry_date"])

print("Raw signals:", len(signals))
display(signals.head())

In [ ]:
# ============================================================
# 10. SIGNAL COOLDOWN
# ============================================================
signals = signals.sort_values(["symbol", "signal_date"]).reset_index(drop=True)

accepted = []
last_signal_date = {}

for row in signals.itertuples(index=False):
    sym = row.symbol
    dt = pd.Timestamp(row.signal_date)

    if sym not in last_signal_date:
        accepted.append(row)
        last_signal_date[sym] = dt
    elif (dt - last_signal_date[sym]).days >= SIGNAL_COOLDOWN_DAYS:
        accepted.append(row)
        last_signal_date[sym] = dt

signals_cd = pd.DataFrame(accepted)

print("Signals before cooldown:", len(signals))
print("Signals after cooldown:", len(signals_cd))
print("Unique symbols:", signals_cd["symbol"].nunique())
display(signals_cd["score"].value_counts().sort_index().rename("signals").to_frame())

## 11. Optional Nifty regime filter

Disabled by default. If enabled, it requires an available Nifty Parquet file and keeps signals only when Nifty is above its 200DMA.

In [ ]:
# ============================================================
# 11. DISCOVER NIFTY FILES
# ============================================================
nifty_candidates = [
    f for f in files
    if "nifty" in f.name.lower() or "nifty" in str(f.parent).lower()
]

print("Nifty candidates:", len(nifty_candidates))
for f in nifty_candidates[:20]:
    print(f)

In [ ]:
# ============================================================
# 12. OPTIONAL NIFTY FILTER
# ============================================================
if USE_REGIME_FILTER and nifty_candidates:
    nf = str(nifty_candidates[0])
    ncols = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [nf]
    ).df()["column_name"].tolist()

    nd = pick_col(ncols, ["date", "trade_date"])
    nc = pick_col(ncols, ["close", "close_price", "last"])

    if nd and nc:
        nf_escaped = nf.replace("'", "''")

        regime = con.execute(f"""
        WITH n AS (
            SELECT
                CAST("{nd}" AS DATE) AS date,
                TRY_CAST("{nc}" AS DOUBLE) AS close
            FROM read_parquet('{nf_escaped}')
        )
        SELECT
            date,
            close,
            AVG(close) OVER (
                ORDER BY date
                ROWS BETWEEN 199 PRECEDING AND CURRENT ROW
            ) AS sma200
        FROM n
        """).df()

        regime["date"] = pd.to_datetime(regime["date"])
        regime["regime_ok"] = regime["close"] > regime["sma200"]

        signals_cd = signals_cd.merge(
            regime[["date", "regime_ok"]].rename(
                columns={"date": "signal_date"}
            ),
            on="signal_date",
            how="left"
        )

        signals_cd["regime_ok"] = signals_cd["regime_ok"].fillna(False)
        signals_cd = signals_cd[signals_cd["regime_ok"]].copy()

        print("Signals after Nifty filter:", len(signals_cd))
    else:
        print("Could not identify Nifty date/close columns.")
else:
    print("Nifty regime filter disabled.")

In [ ]:
# ============================================================
# 13. LOAD PRICE DATA FOR SIGNAL UNIVERSE
# ============================================================
signal_symbols = sorted(
    signals_cd["symbol"].dropna().astype(str).unique().tolist()
)

if not signal_symbols:
    raise ValueError("No signals available after filtering.")

con.execute("DROP TABLE IF EXISTS signal_symbols")
con.execute("CREATE TEMP TABLE signal_symbols(symbol VARCHAR)")
con.executemany(
    "INSERT INTO signal_symbols VALUES (?)",
    [(s,) for s in signal_symbols]
)

prices = con.execute("""
SELECT
    d.date,
    d.symbol,
    d.open,
    d.high,
    d.low,
    d.close,
    d.volume
FROM daily d
JOIN signal_symbols s USING(symbol)
WHERE d.close > 0
ORDER BY d.date, d.symbol
""").df()

prices["date"] = pd.to_datetime(prices["date"])

print("Price rows:", len(prices))
print("Symbols:", prices["symbol"].nunique())
print("Coverage:", prices["date"].min(), "to", prices["date"].max())

In [ ]:
# ============================================================
# 14. PREPARE PRICE GROUPS
# ============================================================
prices = prices.sort_values(["symbol", "date"]).reset_index(drop=True)

prices["sma50"] = (
    prices.groupby("symbol")["close"]
    .transform(lambda s: s.rolling(50, min_periods=50).mean())
)

price_groups = {
    sym: g.reset_index(drop=True)
    for sym, g in prices.groupby("symbol", sort=False)
}

print("Prepared symbols:", len(price_groups))

## 15. Trade simulator — NO HARD INITIAL STOP

Exit rules:
1. Trailing stop: configurable percentage below the highest close since entry.
2. Failure exit: after 20 sessions, if close is below both entry price and 50DMA.
3. Time exit: after 252 sessions.

There is **no fixed -30% initial stop**.

In [ ]:
# ============================================================
# 15. TRADE SIMULATOR
# ============================================================
def simulate_trade(row, pg, trail_pct=TRAILING_STOP_PCT):
    entry_date = pd.Timestamp(row["entry_date"])
    idxs = pg.index[pg["date"] == entry_date].tolist()

    if not idxs:
        return None

    start = idxs[0]
    entry = float(pg.loc[start, "open"])

    if not np.isfinite(entry) or entry <= 0:
        return None

    max_idx = min(start + TIME_EXIT_DAYS, len(pg) - 1)
    mae = 0.0
    mfe = 0.0

    for i in range(start + 1, max_idx + 1):
        op = float(pg.loc[i, "open"])
        hi = float(pg.loc[i, "high"])
        lo = float(pg.loc[i, "low"])
        cl = float(pg.loc[i, "close"])

        mae = min(mae, lo / entry - 1)
        mfe = max(mfe, hi / entry - 1)

        # Only information available before today's open is used.
        prev_highest = max(
            entry,
            float(pg.loc[start:i-1, "close"].max())
        )
        trailing_stop = prev_highest * (1 - trail_pct)

        if op <= trailing_stop:
            return {
                "symbol": row["symbol"],
                "signal_date": row["signal_date"],
                "entry_date": entry_date,
                "entry_price": entry,
                "exit_date": pg.loc[i, "date"],
                "exit_price": op,
                "return": op / entry - 1,
                "mae": mae,
                "mfe": mfe,
                "exit_reason": "TRAILING_STOP",
                "score": row["score"]
            }

        if i - start >= FAILURE_EXIT_DAYS:
            sma50 = pg.loc[i, "sma50"]
            if np.isfinite(sma50) and cl < sma50 and cl < entry:
                return {
                    "symbol": row["symbol"],
                    "signal_date": row["signal_date"],
                    "entry_date": entry_date,
                    "entry_price": entry,
                    "exit_date": pg.loc[i, "date"],
                    "exit_price": op,
                    "return": op / entry - 1,
                    "mae": mae,
                    "mfe": mfe,
                    "exit_reason": "FAILURE_EXIT",
                    "score": row["score"]
                }

    exit_price = float(pg.loc[max_idx, "close"])

    return {
        "symbol": row["symbol"],
        "signal_date": row["signal_date"],
        "entry_date": entry_date,
        "entry_price": entry,
        "exit_date": pg.loc[max_idx, "date"],
        "exit_price": exit_price,
        "return": exit_price / entry - 1,
        "mae": mae,
        "mfe": mfe,
        "exit_reason": "TIME_EXIT",
        "score": row["score"]
    }

trade_rows = []

for row in signals_cd.itertuples(index=False):
    pg = price_groups.get(row.symbol)
    if pg is None:
        continue

    result = simulate_trade(row._asdict(), pg)
    if result:
        trade_rows.append(result)

trades = pd.DataFrame(trade_rows)

print("Trade records:", len(trades))
display(trades.head())

In [ ]:
# ============================================================
# 16. TRADE-LEVEL SUMMARY
# ============================================================
def summarize_trades(df):
    if df.empty:
        return pd.DataFrame()

    wins = df.loc[df["return"] > 0, "return"]
    losses = df.loc[df["return"] <= 0, "return"]

    gross_profit = wins.sum()
    gross_loss = -losses.sum()

    return pd.DataFrame([{
        "trades": len(df),
        "win_rate": (df["return"] > 0).mean(),
        "mean_return": df["return"].mean(),
        "median_return": df["return"].median(),
        "avg_win": wins.mean() if len(wins) else np.nan,
        "avg_loss": losses.mean() if len(losses) else np.nan,
        "profit_factor": gross_profit / gross_loss if gross_loss > 0 else np.inf,
        "mean_mae": df["mae"].mean(),
        "median_mae": df["mae"].median(),
        "mean_mfe": df["mfe"].mean(),
        "2x_rate": (df["mfe"] >= 1).mean(),
        "3x_rate": (df["mfe"] >= 2).mean(),
        "5x_rate": (df["mfe"] >= 4).mean()
    }])

display(summarize_trades(trades))
display(trades["exit_reason"].value_counts().rename("trades").to_frame())

# 17. Portfolio-level backtest

The portfolio:
- starts with ₹10 lakh
- holds at most 20 positions
- uses approximately equal notional per position
- tracks cash
- marks positions to daily close
- enters at next open
- exits according to the no-hard-stop trade engine
- includes configurable slippage and costs

The portfolio equity curve is the primary performance result. Individual trade averages are only diagnostics.

In [ ]:
# ============================================================
# 17. PORTFOLIO BACKTEST
# ============================================================
signals_port = signals_cd.copy()
signals_port["entry_date"] = pd.to_datetime(signals_port["entry_date"])

signals_port = signals_port.sort_values(
    ["entry_date", "score", "symbol"],
    ascending=[True, False, True]
).reset_index(drop=True)

entries_by_date = {
    dt: g
    for dt, g in signals_port.groupby("entry_date", sort=False)
}

price_lookup = prices.set_index(["date", "symbol"])[["open", "close"]]

trade_lookup = {
    (r.symbol, pd.Timestamp(r.entry_date)): r._asdict()
    for r in trades.itertuples(index=False)
}

all_dates = pd.DatetimeIndex(sorted(prices["date"].unique()))

cash = float(INITIAL_CAPITAL)
positions = {}
closed_portfolio_trades = []
equity_rows = []

for current_date in all_dates:

    # A. Exit positions first.
    for symbol, pos in list(positions.items()):
        if pd.Timestamp(pos["exit_date"]) != current_date:
            continue

        key = (current_date, symbol)
        if key not in price_lookup.index:
            continue

        raw_exit = float(price_lookup.loc[key, "open"])

        sell_cost = (
            SLIPPAGE_BPS + BROKERAGE_BPS +
            STT_BPS_SELL + OTHER_COST_BPS
        ) / 10000.0

        net_exit = raw_exit * (1 - sell_cost)
        cash += pos["shares"] * net_exit

        rec = pos.copy()
        rec["exit_price_raw"] = raw_exit
        rec["exit_price_net"] = net_exit
        rec["realized_return"] = (
            net_exit / pos["entry_price_net"] - 1
        )
        rec["exit_date_actual"] = current_date

        closed_portfolio_trades.append(rec)
        del positions[symbol]

    # B. Mark current portfolio.
    equity_before_entries = cash
    for symbol, pos in positions.items():
        key = (current_date, symbol)
        if key in price_lookup.index:
            equity_before_entries += (
                pos["shares"] * float(price_lookup.loc[key, "close"])
            )

    # C. Enter new positions.
    candidates = entries_by_date.get(current_date)
    slots = MAX_POSITIONS - len(positions)

    if candidates is not None and slots > 0:
        for row in candidates.itertuples(index=False):
            if slots <= 0:
                break

            symbol = row.symbol
            if symbol in positions:
                continue

            key = (current_date, symbol)
            if key not in price_lookup.index:
                continue

            trade = trade_lookup.get((symbol, current_date))
            if trade is None:
                continue

            raw_entry = float(price_lookup.loc[key, "open"])

            buy_cost = (
                SLIPPAGE_BPS + BROKERAGE_BPS + OTHER_COST_BPS
            ) / 10000.0

            entry_net = raw_entry * (1 + buy_cost)

            target_notional = equity_before_entries / MAX_POSITIONS
            notional = min(target_notional, cash)
            shares = math.floor(notional / entry_net)

            if shares <= 0:
                continue

            invested = shares * entry_net
            cash -= invested

            positions[symbol] = {
                "symbol": symbol,
                "signal_date": row.signal_date,
                "entry_date": current_date,
                "entry_price_raw": raw_entry,
                "entry_price_net": entry_net,
                "shares": shares,
                "initial_notional": invested,
                "score": row.score,
                "exit_date": trade["exit_date"],
                "planned_exit_price": trade["exit_price"],
                "planned_exit_reason": trade["exit_reason"],
                "mae": trade["mae"],
                "mfe": trade["mfe"]
            }

            slots -= 1

    # D. Daily mark-to-market.
    equity = cash
    for symbol, pos in positions.items():
        key = (current_date, symbol)
        if key in price_lookup.index:
            equity += (
                pos["shares"] *
                float(price_lookup.loc[key, "close"])
            )

    equity_rows.append({
        "date": current_date,
        "equity": equity,
        "cash": cash,
        "open_positions": len(positions)
    })

equity_curve = pd.DataFrame(equity_rows)

print("Trading days:", len(equity_curve))
print("Closed positions:", len(closed_portfolio_trades))
print("Open positions at end:", len(positions))
display(equity_curve.tail())

In [ ]:
# ============================================================
# 18. PORTFOLIO METRICS
# ============================================================
eq = equity_curve.copy()
eq["date"] = pd.to_datetime(eq["date"])
eq = eq.sort_values("date").reset_index(drop=True)

eq["daily_return"] = eq["equity"].pct_change().fillna(0)
eq["running_peak"] = eq["equity"].cummax()
eq["drawdown"] = eq["equity"] / eq["running_peak"] - 1

start_equity = float(eq["equity"].iloc[0])
end_equity = float(eq["equity"].iloc[-1])

days = max((eq["date"].iloc[-1] - eq["date"].iloc[0]).days, 1)
years = days / 365.25

cagr = (
    (end_equity / start_equity) ** (1 / years) - 1
    if start_equity > 0 else np.nan
)

max_dd = float(eq["drawdown"].min())

std = eq["daily_return"].std()
sharpe = (
    eq["daily_return"].mean() / std * np.sqrt(252)
    if std > 0 else np.nan
)

downside = eq.loc[eq["daily_return"] < 0, "daily_return"].std()
sortino = (
    eq["daily_return"].mean() / downside * np.sqrt(252)
    if downside > 0 else np.nan
)

calmar = cagr / abs(max_dd) if max_dd < 0 else np.nan

portfolio_metrics = pd.DataFrame([{
    "start_date": eq["date"].min(),
    "end_date": eq["date"].max(),
    "starting_capital": start_equity,
    "ending_equity": end_equity,
    "CAGR": cagr,
    "max_drawdown": max_dd,
    "Sharpe": sharpe,
    "Sortino": sortino,
    "Calmar": calmar,
    "closed_trades": len(closed_portfolio_trades),
    "max_positions": MAX_POSITIONS,
    "trailing_stop": TRAILING_STOP_PCT,
    "hard_initial_stop": False
}])

display(portfolio_metrics.T)

In [ ]:
# ============================================================
# 19. YEARLY PORTFOLIO PERFORMANCE
# ============================================================
eq["year"] = eq["date"].dt.year

yearly_portfolio = (
    eq.groupby("year")
      .agg(
          start_equity=("equity", "first"),
          end_equity=("equity", "last"),
          avg_open_positions=("open_positions", "mean"),
          max_open_positions=("open_positions", "max")
      )
)

yearly_portfolio["return"] = (
    yearly_portfolio["end_equity"] /
    yearly_portfolio["start_equity"] - 1
)

display(yearly_portfolio)

In [ ]:
# ============================================================
# 20. PORTFOLIO PLOTS
# ============================================================
fig = plt.figure(figsize=(14, 5))
plt.plot(eq["date"], eq["equity"])
plt.title("Portfolio Equity Curve — No Hard Initial Stop")
plt.xlabel("Date")
plt.ylabel("Equity (₹)")
plt.grid(True, alpha=0.25)
plt.show()

fig = plt.figure(figsize=(14, 4))
plt.plot(eq["date"], eq["drawdown"] * 100)
plt.title("Portfolio Drawdown")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.grid(True, alpha=0.25)
plt.show()

fig = plt.figure(figsize=(12, 4))
plt.plot(eq["date"], eq["open_positions"])
plt.title("Concurrent Open Positions")
plt.xlabel("Date")
plt.ylabel("Positions")
plt.grid(True, alpha=0.25)
plt.show()

In [ ]:
# ============================================================
# 21. SCORE ANALYSIS
# ============================================================
score_stats = (
    trades.groupby("score")
    .agg(
        trades=("return", "size"),
        win_rate=("return", lambda x: (x > 0).mean()),
        mean_return=("return", "mean"),
        median_return=("return", "median"),
        avg_mae=("mae", "mean"),
        avg_mfe=("mfe", "mean"),
        two_x=("mfe", lambda x: (x >= 1).mean()),
        three_x=("mfe", lambda x: (x >= 2).mean()),
        five_x=("mfe", lambda x: (x >= 4).mean())
    )
    .reset_index()
)

display(score_stats)

In [ ]:
# ============================================================
# 22. YEARLY TRADE-LEVEL ANALYSIS
# ============================================================
trades["year"] = pd.to_datetime(trades["entry_date"]).dt.year

annual_trade = (
    trades.groupby("year")
    .agg(
        trades=("return", "size"),
        win_rate=("return", lambda x: (x > 0).mean()),
        mean_return=("return", "mean"),
        median_return=("return", "median"),
        avg_mae=("mae", "mean"),
        avg_mfe=("mfe", "mean")
    )
    .reset_index()
)

display(annual_trade)

# 23. Trailing-stop sensitivity

Every tested configuration has **no hard initial stop**.
Only the trailing-stop distance changes.

In [ ]:
# ============================================================
# 23. TRAILING STOP SENSITIVITY
# ============================================================
def simulate_for_trail(row, pg, trail_pct):
    entry_date = pd.Timestamp(row["entry_date"])
    idxs = pg.index[pg["date"] == entry_date].tolist()

    if not idxs:
        return None

    start = idxs[0]
    entry = float(pg.loc[start, "open"])
    max_idx = min(start + TIME_EXIT_DAYS, len(pg) - 1)

    mae = 0.0
    mfe = 0.0

    for i in range(start + 1, max_idx + 1):
        op = float(pg.loc[i, "open"])
        hi = float(pg.loc[i, "high"])
        lo = float(pg.loc[i, "low"])

        mae = min(mae, lo / entry - 1)
        mfe = max(mfe, hi / entry - 1)

        prev_highest = max(
            entry,
            float(pg.loc[start:i-1, "close"].max())
        )

        stop = prev_highest * (1 - trail_pct)

        if op <= stop:
            return {
                "return": op / entry - 1,
                "mae": mae,
                "mfe": mfe
            }

    exit_px = float(pg.loc[max_idx, "close"])

    return {
        "return": exit_px / entry - 1,
        "mae": mae,
        "mfe": mfe
    }

sensitivity = []

for trail in [0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50]:
    rows = []

    for row in signals_cd.itertuples(index=False):
        pg = price_groups.get(row.symbol)
        if pg is None:
            continue

        result = simulate_for_trail(row._asdict(), pg, trail)

        if result:
            rows.append(result)

    sdf = pd.DataFrame(rows)

    if not sdf.empty:
        sensitivity.append({
            "trailing_stop": trail,
            "trades": len(sdf),
            "win_rate": (sdf["return"] > 0).mean(),
            "mean_return": sdf["return"].mean(),
            "median_return": sdf["return"].median(),
            "avg_mae": sdf["mae"].mean(),
            "2x_rate": (sdf["mfe"] >= 1).mean(),
            "3x_rate": (sdf["mfe"] >= 2).mean(),
            "5x_rate": (sdf["mfe"] >= 4).mean()
        })

trail_sensitivity = pd.DataFrame(sensitivity)
display(trail_sensitivity)

# 24. Monte Carlo trade-sequence analysis

This bootstraps the observed trade-return distribution to estimate sequence risk.

It is not an out-of-sample test.

In [ ]:
# ============================================================
# 24. MONTE CARLO
# ============================================================
def monte_carlo(returns, n_paths=2000, seed=42):
    r = np.asarray(returns, dtype=float)
    r = r[np.isfinite(r)]

    if len(r) < 20:
        return pd.DataFrame()

    rng = np.random.default_rng(seed)
    result = []

    for _ in range(n_paths):
        sample = rng.choice(r, size=len(r), replace=True)
        equity_path = np.cumprod(1 + sample)
        peak = np.maximum.accumulate(equity_path)
        dd = equity_path / peak - 1

        result.append({
            "terminal_return": equity_path[-1] - 1,
            "max_drawdown": dd.min()
        })

    return pd.DataFrame(result)

mc = monte_carlo(trades["return"].values)

if not mc.empty:
    display(
        mc.describe(
            percentiles=[0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
        )
    )

    fig = plt.figure(figsize=(10, 5))
    plt.hist(mc["max_drawdown"], bins=60)
    plt.title("Monte Carlo Maximum Drawdown")
    plt.xlabel("Maximum Drawdown")
    plt.ylabel("Count")
    plt.grid(True, alpha=0.25)
    plt.show()

# 25. Walk-forward periods

This creates train / validation / test date ranges.

For actual parameter selection, use only train and validation. Keep the final test period untouched.

In [ ]:
# ============================================================
# 25. WALK-FORWARD PERIODS
# ============================================================
unique_dates = np.array(sorted(pd.to_datetime(signals_cd["entry_date"].unique())))

if len(unique_dates) >= 10:
    n = len(unique_dates)
    train_end = unique_dates[int(n * 0.60)]
    validation_end = unique_dates[int(n * 0.80)]

    walk_forward_periods = pd.DataFrame({
        "period": ["train", "validation", "test"],
        "start": [unique_dates[0], train_end, validation_end],
        "end": [train_end, validation_end, unique_dates[-1]]
    })

    display(walk_forward_periods)
else:
    print("Not enough signal dates for a meaningful split.")

In [ ]:
# ============================================================
# 26. EXPORT RESULTS
# ============================================================
trades.to_parquet(
    RESULTS_DIR / "trade_level_results_no_hard_stop.parquet",
    index=False
)

signals_cd.to_parquet(
    RESULTS_DIR / "signals.parquet",
    index=False
)

eq.to_parquet(
    RESULTS_DIR / "portfolio_equity_no_hard_stop.parquet",
    index=False
)

pd.DataFrame(closed_portfolio_trades).to_parquet(
    RESULTS_DIR / "portfolio_closed_trades.parquet",
    index=False
)

portfolio_metrics.to_csv(
    RESULTS_DIR / "portfolio_metrics_no_hard_stop.csv",
    index=False
)

yearly_portfolio.to_csv(
    RESULTS_DIR / "portfolio_yearly_no_hard_stop.csv"
)

score_stats.to_csv(
    RESULTS_DIR / "score_analysis.csv",
    index=False
)

trail_sensitivity.to_csv(
    RESULTS_DIR / "trailing_stop_sensitivity.csv",
    index=False
)

if not mc.empty:
    mc.to_parquet(
        RESULTS_DIR / "monte_carlo.parquet",
        index=False
    )

print("Results exported to:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*")):
    print(" ", p.name)

# 27. Final validation checklist

Before treating the strategy as investable:

- Verify the Parquet data reaches the intended latest date.
- Verify corporate-action-adjusted prices.
- Verify delisted / merged securities.
- Verify no future information enters the signal.
- Verify next-open execution.
- Confirm there is no hard initial stop.
- Review trailing-stop sensitivity.
- Review portfolio CAGR and maximum drawdown.
- Add realistic liquidity/slippage constraints.
- Test Nifty regime filtering.
- Run rolling walk-forward tests.
- Add fundamentals only after the technical baseline is stable.

**Primary performance output:** the portfolio equity curve and portfolio metrics. Individual trade averages are diagnostics, not portfolio returns.